# CellDIVE QC Pipeline

Thin notebook wrapper around `scripts/qc/`. All QC logic lives in importable modules; this notebook loads config, runs modules, displays summaries, and writes the updated AnnData.

In [1]:
import gc
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.qc.config import load_config
from scripts.qc.io import QCContext, free_memory
from scripts.qc.run_qc import run_module, run_cohort
from scripts.qc.writers import finalize_qc

SLIDE_ID = "SLIDE-045"
MODULES = ["image", "segmentation", "marker", "spatial"]
cfg = load_config(PROJECT_ROOT / "qc_config.yaml")
print(f"low_memory={cfg.raw.get('low_memory')}, pyramid_level={cfg.pyramid_level}, skip_regionprops={cfg.raw.get('skip_regionprops')}")

low_memory=True, pyramid_level=4, skip_regionprops=True


## Run per-slide QC (memory-safe)

Runs one module at a time with garbage collection between steps. Uses `low_memory: true` in `qc_config.yaml` (backed AnnData, sparse pixel sampling, skips `regionprops`).

In [ ]:
import time

ctx = QCContext.from_slide(cfg, SLIDE_ID)
slide = cfg.slide(SLIDE_ID)
module_results = {}

for module in MODULES:
    t_mod = time.perf_counter()
    print(f"Module: {module}")
    module_results[module] = run_module(ctx, module)
    ctx.clear_caches()
    free_memory()
    gc.collect()
    print(f"  -> module finished in {time.perf_counter() - t_mod:.1f}s\n")

ctx.apply_cell_flags_to_adata()
summary = finalize_qc(ctx.adata, slide, cfg, module_results)

out_path = slide.adata_out or slide.adata_in
ctx.adata.write_h5ad(out_path)
print(f"Wrote {out_path}")

summary["overall_status"], summary["n_cells_qc_pass"], summary["n_cells"]

Module: image
  [image] focus...
  [image] illumination...
  [image] registration_drift...
  [image] bleedthrough...


/home/steve/miniforge3/envs/napari-env/lib/python3.11/site-packages/skimage/registration/_phase_cross_correlation.py:119: RuntimeWarning: overflow encountered in scalar multiply
  amp = src_amp * target_amp


  [image] autofluorescence...
Module: segmentation
  [segmentation] mask_vs_dapi...
  [segmentation] area_shape...
  [segmentation] tile_outliers...
  [segmentation] coverage...
Module: marker
  [marker] signal_quality...
  [marker] dapi_correlation...
  [marker] cross_marker_correlation...


## Module status summary

In [ ]:
rows = []
for mod, block in summary["modules"].items():
    rows.append({"module": mod, "status": block["status"]})
display(pd.DataFrame(rows))

## Marker QC table

In [ ]:
marker_tsv = cfg.qc_dir / SLIDE_ID / "marker_qc.tsv"
marker_df = pd.read_csv(marker_tsv, sep="\t")
display(marker_df.sort_values("staining_index", ascending=False))

## Key figures

In [ ]:
fig_dir = cfg.qc_dir / SLIDE_ID / "figures"
for name in [
    "mask_vs_dapi_qc.png",
    "marker_staining_index.png",
    "density_map.png",
    "focus_heatmap.png",
    "drift_dapi_rounds.png",
]:
    path = fig_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## Cohort / batch QC (optional; needs >=2 slides)

In [ ]:
cohort = run_cohort(cfg)
cohort

## Render Quarto report

From a terminal:

```bash
quarto render scripts/qc/report/qc_report.qmd -P slide_id:SLIDE-045
```